# Tumour spheroid

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sisyga/biolgca/blob/aidevelop/docs/source/examples/tumour_spheroid.ipynb)

A few thousand tumour cells in culture grow into a ball, a multicellular
spheroid. Oxygen reaches its cells only by diffusion from the medium, and
they consume it on the way in. Once the spheroid is about half a millimetre
across, the centre runs out: cells there stop dividing and then die. The
spheroid grows on as a necrotic core, a ring of quiescent cells and a rim of
proliferating cells, much as a tumour does before it has blood vessels.

This advanced example builds such a spheroid from the mechanisms of the
earlier lessons: three classes of cells as the species of one model, a field
that the cells consume, switches between the classes that respond to the
field, and growth limited by room. It follows the hybrid cellular automaton
of Dormann & Deutsch (2002) in outline, with the parameters of measurements
made since: the nutrient is oxygen, with its measured diffusion, consumption
and thresholds, and quiescent cells are a class of their own. The model runs
as a cross-section on a hexagonal lattice and as the spheroid itself on the
3D Moore lattice, and its cells of each node are drawn by their mean oxygen
uptake, a property of their classes.

**Question.** How does the supply of oxygen layer a growing spheroid, and do
measured parameters give the measured layers?

In [ ]:
# In Google Colab this cell installs BioLGCA (about a minute); elsewhere it does nothing.
import importlib.util
import subprocess
import sys

if "google.colab" in sys.modules and importlib.util.find_spec("lgca") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "biolgca @ git+https://github.com/sisyga/biolgca@aidevelop"], check=True)

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

from lgca.examples import tumour_spheroid as ts
from lgca.model import build_model, run_model

## Parameters from measurements

The parameters are those of `ts.Parameters`, whose documentation gives the
sources; times in hours, lengths in µm, oxygen as a partial pressure in mmHg
(1.3 µM per mmHg at 37 °C).

| Quantity | Value | Measured | Source |
|---|---|---|---|
| Doubling time of proliferating cells | 20 h | 20–24 h, EMT6/Ro spheroids; 22 h, HCT116 | Freyer & Sutherland 1986; Mao et al. 2018 |
| Volume of a cell | 1.4 pL | DLD1 (radius 7 µm); HCT116 1.2 pL | Grimes et al. 2014; Mao et al. 2018 |
| Oxygen diffusion | 2000 µm²/s | 1750 µm²/s in tumour tissue | Grote et al. 1977; Grimes et al. 2014 |
| Oxygen of the medium | 140 mmHg | 141 mmHg, air with 5 % CO₂ at 37 °C | Wenger et al. 2015 |
| Unstirred medium around the spheroid | 90 µm | up to about 100 µm | Mueller-Klieser & Sutherland 1982; Grimes et al. 2014 |
| Consumption of proliferating tissue | 22 mmHg/s | DLD1 spheroids; 11–28 mmHg/s, four other lines | Grimes et al. 2014, 2016 |
| Michaelis–Menten constant | 1 mmHg | at most about 1 mmHg | Grimes et al. 2014 |
| Consumption of quiescent cells | 0.4 | 2.5 times less, EMT6/Ro; 5 times less, V79 | Freyer & Sutherland 1985; Freyer et al. 1984 |
| Quiescence below | 5 mmHg | 0.5–5 mmHg, depending on glucose | Grimes et al. 2016, after Höckel & Vaupel 2001 |
| Necrosis below | 0.5 mmHg, in 48 h | 0.8 mmHg severe hypoxia; HCT116 die a day after a day below 0.1 mmHg | Grimes et al. 2014; Mao et al. 2018 |
| Arrest, recovery | 4 h, 10 h | DNA synthesis resumes hours after reoxygenation | assumed; Amellem & Pettersen 1991 |
| Lysis of necrotic material | 2000 h | not measured | Dormann & Deutsch 2002 |

A node is a voxel of tissue 30 µm across, which holds about a dozen cells
(65 % of its volume): a model without volume exclusion, whose capacity is
the number of cells that fit. One step is one hour. In these units:

In [ ]:
parameters = ts.Parameters()
for geometry in ("hex", "moore"):
    lattice = ts.lattice_parameters(parameters, geometry)
    print(f"{geometry:5s}: capacity {lattice['capacity']} cells, D = {lattice['diffusion']:.0f} nodes²/h, "
          f"uptake {lattice['uptake']:.1f} per cell and hour, division {lattice['division']:.3f} per hour")

# the distance over which tissue uses up the oxygen of the medium
penetration = math.sqrt(parameters.oxygen_diffusion * parameters.medium_oxygen / parameters.uptake)
print(f"penetration length {penetration:.0f} µm = {penetration / parameters.node_size:.1f} nodes")
print(f"oxygen runs out at the centre of a ball of radius {math.sqrt(6) * penetration:.0f} µm "
      f"and of a cylinder of radius {math.sqrt(4) * penetration:.0f} µm")

Oxygen diffuses $D \approx 8000$ node spacings squared per hour, so it
crosses the spheroid in minutes, while cells divide in a day: at every step
the field is at equilibrium with the cells (the steady solver). The
penetration length $\lambda = \sqrt{D c_0 / q}$, for the medium's oxygen
$c_0$ and the consumption $q$ of tissue, is about four nodes, so the field's
gradient is resolved. Were all cells to consume at the full rate,
the centre of a ball would run out of oxygen at radius $\sqrt{6}\,\lambda$,
that of a cylinder at $\sqrt{4}\,\lambda$: a cross-section, whose oxygen
comes in from the sides only, starves earlier than the spheroid.

## The model

Every hour:

1. `tumour_spheroid.stir` marks the stirred medium: the nodes farther than
   the unstirred layer from every cell. Their oxygen stays the medium's
   (the reaction `tumour_spheroid.medium`).
2. `pde` solves the oxygen at equilibrium with the cells: proliferating
   cells (species 0) consume $q\,c / (K_m + c)$ each, quiescent ones
   (species 1) 0.4 times as much, necrotic material (species 2) nothing.
3. `phenotype_switch`: below 5 mmHg proliferating cells become quiescent
   (in 4 h on average), above it quiescent cells divide again (in 10 h),
   and below 0.5 mmHg both die (in 48 h). The thresholds are Hill
   responses to the oxygen, steep but not sharp.
4. `birth_death`: proliferating cells divide where oxygen allows, until
   their node is full; necrotic material dissolves slowly.
5. `go_or_rest` and `random_walk`: the living cells of full nodes move to
   neighbouring nodes; the others stay. This is how division pushes the
   spheroid outward.

`ts.build_spec` writes it as a `ModelSpec`:

In [ ]:
spec = ts.build_spec("hex", steps=240)  # 80 x 80 nodes, 2.4 mm across; ten days
for operator in spec.dynamics.operators:
    print(operator["name"] if isinstance(operator, dict) else operator.name)

## A cross-section on the hexagonal lattice

The cross-section starts as a disc of 100 µm radius filled with
proliferating cells and grows for ten days; the cells per class and the
oxygen are recorded every day.

In [ ]:
cross_section = run_model(spec, showprogress=False)
density = np.asarray(cross_section.data["density"])  # day, nodes, class
oxygen = np.asarray(cross_section.data["oxygen"]) * parameters.medium_oxygen  # mmHg
days = np.asarray(cross_section.data.steps("density")) / 24
lattice = cross_section.lgca
middle = (lattice.xcoords.max() / 2, lattice.ycoords.max() / 2)


def crop(axis):
    # the middle of the lattice, where the cross-section grows
    axis.set(xlim=(middle[0] - 26, middle[0] + 26), ylim=(middle[1] - 26, middle[1] + 26), xticks=[], yticks=[],
             xlabel="", ylabel="")


fig, axes = plt.subplots(1, 5, figsize=(13, 3), constrained_layout=True)
for axis, day in zip(axes, (2, 4, 6, 8, 10)):
    uptake = ts.mean_uptake(None, density[day])
    lattice.plot_scalarfield(uptake, ax=axis, mask=np.isnan(uptake), cmap="viridis", vmin=0, vmax=1, cbar=False,
                             edgecolor="none")
    axis.set_title(f"day {day}")
    crop(axis)
fig.colorbar(plt.cm.ScalarMappable(plt.Normalize(0, 1), "viridis"), ax=axes, shrink=0.8,
             label="mean uptake per cell")
plt.show()
plt.close(fig)

Each node shows the mean oxygen uptake of its cells, relative to a
proliferating cell: 1 where all proliferate, 0.4 where all are quiescent,
0 for necrotic material (`ts.mean_uptake`, which calls
`lgca.plot_data.mean_species_property` with one value per class). For two
days the disc grows as a whole. Then its centre turns quiescent, and from
the third day a necrotic core grows behind a quiescent ring and a bright
rim of proliferating cells. Its outline follows the six directions of the
hexagonal lattice a little, along which cells move.

On the last day, the oxygen, and averaged over rings around the centre,
where each class lives and the oxygen that puts it there:

In [ ]:
final = density[10]
cells = final.sum(-1)
fractions = [np.where(cells > 0, final[..., k] / np.maximum(cells, 1), np.nan) for k in range(3)]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True, width_ratios=(0.8, 1, 1))
lattice.plot_scalarfield(oxygen[10], ax=axes[0], cmap="magma", vmin=0, vmax=parameters.medium_oxygen,
                         cbar=False, edgecolor="none")
crop(axes[0])
axes[0].set_title("oxygen, day 10")
fig.colorbar(plt.cm.ScalarMappable(plt.Normalize(0, parameters.medium_oxygen), "magma"), ax=axes[0],
             location="bottom", shrink=0.8, label="oxygen (mmHg)")
for k, (name, colour) in enumerate(zip(ts.CLASSES, ("tab:orange", "tab:blue", "black"))):
    distance, profile = ts.radial_profile(fractions[k], "hex", final)
    axes[1].plot(distance * parameters.node_size, profile, color=colour, label=name)
distance, profile = ts.radial_profile(oxygen[10], "hex", final)
axes[2].semilogy(distance * parameters.node_size, profile, color="tab:red")
for threshold, label in ((parameters.quiescence_oxygen, "quiescence"), (parameters.necrosis_oxygen, "necrosis")):
    axes[2].axhline(threshold, color="grey", linestyle=":")
    axes[2].text(10, threshold * 1.15, label, color="grey")
axes[1].set(xlabel="distance from the centre (µm)", ylabel="fraction of the cells", xlim=(0, 900))
axes[2].set(xlabel="distance from the centre (µm)", ylabel="oxygen (mmHg)", xlim=(0, 900), ylim=(0.05, 200))
axes[1].legend()
plt.show()
plt.close(fig)

The boundaries of the classes follow the thresholds of the oxygen, but not
sharply: cells take hours to stop dividing and two days to die, and the
tissue moves meanwhile. Quiescent cells therefore reach into the
proliferating rim, and living cells into the core, where they turn necrotic
one by one, as in the transition zones of real spheroids.

The radius of each layer follows from the cells of its classes, taken as
concentric discs of full tissue (`ts.layers`):

In [ ]:
radii = ts.layers(density, "hex")
counts = density.sum(axis=(1, 2))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
for name, label in (("spheroid", "cross-section"), ("quiescent", "quiescent ring and core"),
                    ("necrotic", "necrotic core")):
    axes[0].plot(days, radii[name], marker="o", label=label)
for k, (name, colour) in enumerate(zip(ts.CLASSES, ("tab:orange", "tab:blue", "black"))):
    axes[1].plot(days, counts[:, k], marker="o", color=colour, label=name)
axes[0].set(xlabel="day", ylabel="radius (µm)")
axes[1].set(xlabel="day", ylabel="cells")
axes[0].legend()
axes[1].legend()
plt.show()
plt.close(fig)
speed = np.diff(radii["spheroid"])[-4:].mean() / 24
print(f"the radius grows {speed:.1f} µm/h over the last four days")

The cross-section first grows exponentially, then its radius grows at a
constant speed: only the rim divides, and its thickness is set by the
oxygen, so the rim adds the same tissue every day. Spheroids of eight cell
lines grow this way, linearly after a short exponential phase, with a
proliferating crust of constant thickness (Conger & Ziskin 1983).

## The spheroid in 3D

On the Moore lattice the same model is the spheroid itself, in a cube of
40³ nodes (1.2 mm across). It takes about two minutes for a week. We step
the model ourselves, in three parts, and record the cells of every class
and the lowest oxygen every six hours. The first three days:

In [ ]:
spheroid = build_model(ts.build_spec("moore", steps=168, record_every=168))
lattice3d = spheroid.lgca
history, lowest_oxygen, sections = {}, {}, {}


def record(hour):
    # the cells of every class and the lowest oxygen; the middle plane every day
    history[hour] = lattice3d.species_density[lattice3d.nonborder].copy()
    oxygen = lattice3d.oxygen[lattice3d.nonborder] * parameters.medium_oxygen
    lowest_oxygen[hour] = oxygen.min()
    if hour % 24 == 0:
        middle = lattice3d.dims[2] // 2
        sections[hour] = (ts.mean_uptake(None, history[hour])[:, :, middle], oxygen[:, :, middle])


def advance(hours):
    start = max(history)
    for hour in range(start + 1, start + hours + 1):
        spheroid.step()
        if hour % 6 == 0:
            record(hour)


record(0)
advance(72)
print(f"day 3: {history[72].sum():.0f} cells, lowest oxygen {lowest_oxygen[72]:.1f} mmHg")

Two more days:

In [ ]:
advance(48)
print(f"day 5: {history[120].sum():.0f} cells, lowest oxygen {lowest_oxygen[120]:.2f} mmHg")

And the last two, with the middle plane of the spheroid on days 3, 5 and 7:

In [ ]:
advance(48)

extent = (0, lattice3d.dims[0] * parameters.node_size / 1000) * 2
fig, axes = plt.subplots(2, 3, figsize=(10.5, 6.6), constrained_layout=True)
for column, day in enumerate((3, 5, 7)):
    uptake, oxygen = sections[24 * day]
    axes[0, column].imshow(uptake.T, origin="lower", cmap="viridis", vmin=0, vmax=1, extent=extent)
    axes[1, column].imshow(oxygen.T, origin="lower", cmap="magma", vmin=0, vmax=parameters.medium_oxygen,
                           extent=extent)
    axes[0, column].set_title(f"day {day}")
for axis in axes.flat:
    axis.set(xticks=[], yticks=[])
axes[1, 0].set(xticks=[0, 0.5, 1], xlabel="mm")
fig.colorbar(plt.cm.ScalarMappable(plt.Normalize(0, 1), "viridis"), ax=axes[0], shrink=0.8,
             label="mean uptake per cell")
fig.colorbar(plt.cm.ScalarMappable(plt.Normalize(0, parameters.medium_oxygen), "magma"), ax=axes[1],
             shrink=0.8, label="oxygen (mmHg)")
plt.show()
plt.close(fig)

On day three the spheroid, about 0.4 mm across, proliferates throughout,
with less oxygen at its centre. On day five its centre is quiescent and
anoxic, and the first cells there have died; on day seven it has the
layers of the cross-section. Seen from outside, with an eighth of the ball
cut away:

In [ ]:
uptake = ts.mean_uptake(None, history[168])
shown = np.isfinite(uptake)
middle = np.array(uptake.shape) / 2
x, y, z = np.indices(uptake.shape)
shown &= ~((x >= middle[0]) & (y >= middle[1]) & (z >= middle[2]))  # the cut

fig = plt.figure(figsize=(6, 5.5))
axis = fig.add_subplot(projection="3d")
points = axis.scatter(x[shown], y[shown], z[shown], c=uptake[shown], cmap="viridis", vmin=0, vmax=1,
                      marker="s", s=10, depthshade=True)
axis.set_box_aspect((1, 1, 1))
axis.view_init(elev=20, azim=40)
axis.set_axis_off()
fig.colorbar(points, ax=axis, label="mean uptake per cell", shrink=0.6)
plt.show()
plt.close(fig)

## Compared with measurements

From the recordings: the doubling time while the spheroid is small, its
diameter when its centre runs out of oxygen (below 0.5 mmHg) and when the
first percent of its cells are necrotic, how deep oxygen reaches into it
after a week, and the thickness of its proliferating rim.

In [ ]:
hours = np.array(sorted(history))
recorded = np.array([history[hour] for hour in hours])
radii = ts.layers(recorded, "moore")
counts = recorded.sum(axis=(1, 2, 3))
total = counts.sum(-1)
diameter = 2 * radii["spheroid"]

anoxic = diameter[np.argmax(np.array([lowest_oxygen[hour] for hour in hours]) < parameters.necrosis_oxygen)]
necrotic = diameter[np.argmax(counts[:, ts.NECROTIC] > 0.01 * total)]
# how deep oxygen reaches: from the surface (where nodes are less than half full) to where it falls below
# 0.5 mmHg, in rings around the centre
oxygen = lattice3d.oxygen[lattice3d.nonborder] * parameters.medium_oxygen
distance, profile = ts.radial_profile(oxygen, "moore", history[168])
_, filling = ts.radial_profile(history[168].sum(-1), "moore", history[168])
surface = distance[np.flatnonzero(filling >= lattice3d.capacity / 2)[-1]]
depth = (surface - distance[np.argmax(profile > parameters.necrosis_oxygen)]) * parameters.node_size
doubling = 24 * math.log(2) / math.log(total[4] / total[0])

rows = [
    ("doubling time, first day", f"{doubling:.0f} h", "20–24 h (EMT6/Ro), 22 h (HCT116)"),
    ("diameter when the centre runs out of oxygen", f"{anoxic:.0f} µm", "about 470 µm (DLD1)"),
    ("diameter at the first necrosis", f"{necrotic:.0f} µm", "a necrotic core above about 500 µm"),
    ("depth that oxygen reaches, day 7", f"{depth:.0f} µm", "188 µm (V79), 220 µm (EMT6/Ro)"),
    ("proliferating rim, day 7", f"{radii['spheroid'][-1] - radii['quiescent'][-1]:.0f} µm", "of constant thickness"),
    ("growth of the radius, days 5 to 7", f"{np.diff(radii['spheroid'])[-8:].mean() / 6:.1f} µm/h",
     "linear after an exponential phase"),
]
width = max(len(row[0]) for row in rows)
for quantity, model_value, measured in rows:
    print(f"{quantity:{width}s}  {model_value:>8s}   {measured}")

The measurements: Freyer & Sutherland (1986) and Mao et al. (2018) for the
doubling time; Grimes et al. (2014), whose measurements on DLD1 spheroids put
the diameter above which their centre runs out of oxygen at about 470 µm;
the review of Weiswald et al.
(2015) for the necrotic core; Mueller-Klieser & Sutherland (1982), who
measured the oxygen falling steeply over 188 µm (V79) and 220 µm (EMT6/Ro)
from the surface of spheroids, as deep as their viable rim; Conger & Ziskin
(1983) for the growth.

The model's numbers come from parameters measured independently of these
results, and they agree. Necrosis follows the anoxia of the centre by the
two days that cells take to die, while the spheroid grows on. The
cross-section starves earlier than the spheroid, at a smaller radius, as the
penetration length predicted.

Only oxygen limits these cells. In spheroids necrosis needs glucose to run
short as well (Hlatky et al. 1988), and EMT6/Ro and V79 spheroids turn
necrotic while their mean oxygen is still 40–60 mmHg
(Mueller-Klieser & Sutherland 1982): oxygen alone gets the length scales
right, not every cause of death.

## Differences from Dormann & Deutsch (2002)

- **Nutrient.** Dormann & Deutsch used glucose and refilled it outside the
  tumour every hour. Here it is oxygen, at equilibrium at every step, and
  the medium keeps its oxygen beyond an unstirred layer.
- **Classes.** Their cells were tumour cells or necrotic material, and a
  quiescent ring appeared where division stopped. Here quiescent cells are a
  class of their own, which consumes less oxygen and divides again when it
  returns.
- **Cells and lattice.** Their square lattice had at most five cells per
  node of 40 µm, with a cell volume 15 to 30 times that of tumour cells.
  Here a node of 30 µm holds a dozen cells of measured volume, on a
  hexagonal or a 3D lattice.
- **Growth plateau.** Their spheroids stopped growing at about 2 mm because
  cells moved up a signal released by necrotic cells, a mechanism that
  later work disputed (McElwain & Pettet 1993; Freyer 1988 found such
  factors to inhibit growth rather than attract). This model has no signal,
  and grows linearly for as long as it runs.

## Things to try

1. **Physiological oxygen.** Tissue has about 5 % oxygen, not the 21 % of
   air: `ts.build_spec("hex", parameters=ts.Parameters(medium_oxygen=38))`.
   How large does the cross-section grow in ten days, and when does its core
   die? Gomes et al. (2016) found HCT116 spheroids to grow far more slowly
   at 5 % oxygen.
2. **Glucose.** With plenty of glucose, cells keep dividing down to about
   0.5 mmHg (`quiescence_oxygen=0.5`). What remains of the quiescent ring?
3. **Another cell line.** Quiescent V79 cells consume five times less
   oxygen than proliferating ones (`quiescent_uptake=0.2`). Is their viable
   rim thicker?
4. **A static culture.** Without stirring, the unstirred layer is thicker
   (`unstirred_layer=300`). Where does the spheroid now get its oxygen?
5. **Dormann & Deutsch's signal.** Let necrotic material secrete a field
   (a second `PDESpec` with `cells=[{"production": 1.0, "species": 2}]`)
   and let living cells move up its gradient (a `ReorientationSpec` with
   the `chemotaxis` term). Does the spheroid stop growing?

## References

- Amellem Ø, Pettersen EO (1991) Cell inactivation and cell cycle inhibition as induced by extreme hypoxia: the possible role of cell cycle arrest as a protection against hypoxia-induced lethal damage. *Cell Prolif* 24:127–141. [doi:10.1111/j.1365-2184.1991.tb01144.x](https://doi.org/10.1111/j.1365-2184.1991.tb01144.x)
- Conger AD, Ziskin MC (1983) Growth of mammalian multicellular tumor spheroids. *Cancer Res* 43:556–560. PMID 6848179
- Dormann S, Deutsch A (2002) Modeling of self-organized avascular tumor growth with a hybrid cellular automaton. *In Silico Biol* 2:393–406. PMID 12542422
- Freyer JP (1988) Role of necrosis in regulating the growth saturation of multicellular spheroids. *Cancer Res* 48:2432–2439. PMID 3356007
- Freyer JP, Sutherland RM (1985) A reduction in the in situ rates of oxygen and glucose consumption of cells in EMT6/Ro spheroids during growth. *J Cell Physiol* 124:516–524. [doi:10.1002/jcp.1041240323](https://doi.org/10.1002/jcp.1041240323)
- Freyer JP, Sutherland RM (1986) Regulation of growth saturation and development of necrosis in EMT6/Ro multicellular spheroids by the glucose and oxygen supply. *Cancer Res* 46:3504–3512. PMID 3708582
- Freyer JP, Tustanoff E, Franko AJ, Sutherland RM (1984) In situ oxygen consumption rates of cells in V-79 multicellular spheroids during growth. *J Cell Physiol* 118:53–61. [doi:10.1002/jcp.1041180111](https://doi.org/10.1002/jcp.1041180111)
- Gomes A, Guillaume L, Grimes DR, Fehrenbach J, Lobjois V, Ducommun B (2016) Oxygen partial pressure is a rate-limiting parameter for cell proliferation in 3D spheroids grown in physioxic culture condition. *PLoS One* 11:e0161239. [doi:10.1371/journal.pone.0161239](https://doi.org/10.1371/journal.pone.0161239)
- Grimes DR, Kelly C, Bloch K, Partridge M (2014) A method for estimating the oxygen consumption rate in multicellular tumour spheroids. *J R Soc Interface* 11:20131124. [doi:10.1098/rsif.2013.1124](https://doi.org/10.1098/rsif.2013.1124)
- Grimes DR, Fletcher AG, Partridge M (2014) Oxygen consumption dynamics in steady-state tumour models. *R Soc Open Sci* 1:140080. [doi:10.1098/rsos.140080](https://doi.org/10.1098/rsos.140080)
- Grimes DR, Kannan P, McIntyre A, et al. (2016) The role of oxygen in avascular tumor growth. *PLoS One* 11:e0153692. [doi:10.1371/journal.pone.0153692](https://doi.org/10.1371/journal.pone.0153692)
- Grote J, Süsskind R, Vaupel P (1977) Oxygen diffusivity in tumor tissue (DS-carcinosarcoma) under temperature conditions within the range of 20–40 °C. *Pflügers Arch* 372:37–42. [doi:10.1007/BF00582204](https://doi.org/10.1007/BF00582204)
- Hlatky L, Sachs RK, Alpen EL (1988) Joint oxygen–glucose deprivation as the cause of necrosis in a tumor analog. *J Cell Physiol* 134:167–178. [doi:10.1002/jcp.1041340202](https://doi.org/10.1002/jcp.1041340202)
- Höckel M, Vaupel P (2001) Tumor hypoxia: definitions and current clinical, biologic, and molecular aspects. *J Natl Cancer Inst* 93:266–276. [doi:10.1093/jnci/93.4.266](https://doi.org/10.1093/jnci/93.4.266)
- Mao X, McManaway S, Jaiswal JK, Patel PB, Wilson WR, Hicks KO (2018) An agent-based model for drug-radiation interactions in the tumour microenvironment: hypoxia-activated prodrug SN30000 in multicellular tumour spheroids. *PLoS Comput Biol* 14:e1006469. [doi:10.1371/journal.pcbi.1006469](https://doi.org/10.1371/journal.pcbi.1006469)
- McElwain DLS, Pettet GJ (1993) Cell migration in multicell spheroids: swimming against the tide. *Bull Math Biol* 55:655–674. [doi:10.1007/BF02460655](https://doi.org/10.1007/BF02460655)
- Mueller-Klieser WF, Sutherland RM (1982) Oxygen tensions in multicell spheroids of two cell lines. *Br J Cancer* 45:256–264. [doi:10.1038/bjc.1982.41](https://doi.org/10.1038/bjc.1982.41)
- Weiswald LB, Bellet D, Dangles-Marie V (2015) Spherical cancer models in tumor biology. *Neoplasia* 17:1–15. [doi:10.1016/j.neo.2014.12.004](https://doi.org/10.1016/j.neo.2014.12.004)
- Wenger RH, Kurtcuoglu V, Scholz CC, Marti HH, Hoogewijs D (2015) Frequently asked questions in hypoxia research. *Hypoxia* 3:35–43. [doi:10.2147/HP.S92198](https://doi.org/10.2147/HP.S92198)